# 🚁 Drone Flight Plan — train the field-detection model on Colab
Runtime → Change runtime type → **T4 GPU**, then run all.

Everything (data, training, metrics) comes from the `drone_flight_plan` package — this notebook only calls it.
Models + reports are saved to Google Drive, and training **resumes automatically** after a disconnect.

In [ ]:
#@title Settings
REPO_URL = ""            #@param {type:"string"}
# e.g. https://github.com/<you>/drone_flight_plan.git  — leave empty to install from Drive folder below
DRIVE_CODE_DIR = "/content/drive/MyDrive/drone_flight_plan-pkg"   #@param {type:"string"}
PROFILE = "default"      #@param ["default", "india_mapping", "india_spraying"]
SYNTHETIC = False        #@param {type:"boolean"}
ARCH = "unet"            #@param ["unet", "deeplabv3plus"]
EPOCHS = 25              #@param {type:"integer"}
N_TRAIN = 1500           #@param {type:"integer"}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if REPO_URL:
    !pip -q install "drone-flight-plan[ml] @ git+{REPO_URL}"
else:
    !pip -q install -e "{DRIVE_CODE_DIR}[ml]"
OUT = "/content/drive/MyDrive/dfp_runs"
os.makedirs(OUT, exist_ok=True)

In [ ]:
from drone_flight_plan import load_config
cfg = load_config(PROFILE, {
    "segmentation": {"arch": ARCH},
    "train": {"data_dir": "/content/data/ai4boundaries",      # fast local disk
              "out_dir": f"{OUT}/models", "epochs": EPOCHS, "n_train": N_TRAIN},
})
print(cfg.segmentation, cfg.train, sep="\n")

## 1 · Data

In [ ]:
from drone_flight_plan.segmentation.data import download_ai4boundaries, make_synthetic_dataset
meta = make_synthetic_dataset(cfg.train) if SYNTHETIC else download_ai4boundaries(cfg.train)
meta.split.value_counts()

## 2 · Train (resumes if interrupted)

In [ ]:
from drone_flight_plan.segmentation.train import train
res = train(cfg, meta)
res["best_miou"], res["checkpoint"]

## 3 · Evaluate: tune thresholds on validation → test metrics → end-to-end planning score

In [ ]:
import json, pandas as pd
from drone_flight_plan.segmentation.infer import Predictor
from drone_flight_plan.segmentation.metrics import predict_split, tune_thresholds, evaluate
from drone_flight_plan.evaluation.e2e import end_to_end
pred = Predictor(res["checkpoint"], cfg.segmentation)
best, grid = tune_thresholds(predict_split(pred, meta[meta.split == "val"], 100))
cfg.segmentation.seed_thr, cfg.segmentation.boundary_thr = best["seed_thr"], best["boundary_thr"]
print("tuned:", best)
cache = predict_split(pred, meta[meta.split == "test"])
pixel, summary = evaluate(cache, cfg.segmentation.seed_thr, cfg.segmentation.boundary_thr)
display(pixel); display(pd.Series(summary))
e2e = end_to_end(cache[:40], load_config(PROFILE, {"mission": {"work_agl_m": 20}}), extra_margins=(0, 2, 4))
display(e2e.groupby("extra_margin_m")[["polygon_IoU", "coverage_ai_plan", "outside_ai_plan_m", "fully_inside"]].mean().round(3))
os.makedirs(f"{OUT}/reports", exist_ok=True)
pixel.to_csv(f"{OUT}/reports/pixel_metrics.csv"); e2e.to_csv(f"{OUT}/reports/end_to_end.csv", index=False)
json.dump({**summary, "thresholds": best}, open(f"{OUT}/reports/summary.json", "w"), indent=2)

## 4 · Look at predictions

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from drone_flight_plan.segmentation.postprocess import probs_to_instances
fig, ax = plt.subplots(3, 3, figsize=(15, 15))
for row, (fid, rgb, tr, crs, tgt, enum, prob, _) in zip(ax, cache[:3]):
    lab = probs_to_instances(prob, cfg.segmentation.seed_thr, cfg.segmentation.boundary_thr)
    row[0].imshow(rgb.transpose(1, 2, 0)); row[0].set_title(fid)
    row[1].imshow(prob.argmax(0), vmin=0, vmax=2); row[1].set_title("bg / field / boundary")
    row[2].imshow(rgb.transpose(1, 2, 0)); row[2].imshow(np.ma.masked_equal(lab, 0) % 20, cmap="tab20", alpha=.5)
    row[2].set_title(f"{lab.max()} fields")
[a.axis("off") for a in ax.ravel()]; plt.tight_layout(); plt.savefig(f"{OUT}/reports/predictions.png", dpi=100)

## 5 · Use the model
Copy `dfp_runs/models/<arch>_<encoder>_3class.pt` to your PC and set its path in the app sidebar,
or run: `drone-flight-plan segment area.tif --checkpoint <path>`.

## 6 · Fine-tune for Indian fields (when you have your own labelled tiles)
1. Fly mapping missions → orthomosaic in WebODM → cut into 512×512 tiles.
2. Draw field polygons in QGIS (or CVAT) → rasterise to a **field-ID mask** per tile (0 = background, 1..N = fields).
3. Put them in `images/<id>.tif` and `masks/<id>.tif`, then run the cell below.

In [ ]:
from drone_flight_plan.segmentation.data import meta_from_folder
MY_DATA = "/content/drive/MyDrive/india_tiles"     # images/ + masks/
meta_in = meta_from_folder(MY_DATA)
cfg_ft = load_config(PROFILE, {"segmentation": {"arch": ARCH}, "train": {
    "out_dir": f"{OUT}/models_india", "epochs": 15, "lr": 1e-4, "resume": True,
    "init_checkpoint": res["checkpoint"]}})            # start from the EU-trained model
res_ft = train(cfg_ft, meta_in)
pred_ft = Predictor(res_ft["checkpoint"], cfg_ft.segmentation)
display(evaluate(predict_split(pred_ft, meta_in[meta_in.split == "test"]))[1])